# WideResNet-28-4 + SE Attention — CIFAR-100 (Colab)
**Runtime → Change runtime type → T4 GPU.** Estimated: ~2-3 hours.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt

NUM_CLASSES = 100
BATCH_SIZE = 128
EPOCHS = 200
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
DROPOUT = 0.3
DEPTH = 28
WIDEN_FACTOR = 4
CUTOUT_SIZE = 8
LABEL_SMOOTHING = 0.1

DRIVE_DIR = '/content/drive/MyDrive/cifar100_wrn'
os.makedirs(DRIVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Model

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        b, c, _, _ = x.shape
        w = F.adaptive_avg_pool2d(x, 1).view(b, c)
        w = F.relu(self.fc1(w), inplace=True)
        w = torch.sigmoid(self.fc2(w)).view(b, c, 1, 1)
        return x * w


class WideResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.3):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.se = SEBlock(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, 1, stride, bias=False)

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x), inplace=True))
        out = self.dropout(out)
        out = self.conv2(F.relu(self.bn2(out), inplace=True))
        out = self.se(out)
        return out + self.shortcut(x)


class WideResNet(nn.Module):
    def __init__(self, depth=28, widen_factor=4, dropout=0.3, num_classes=100):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n = (depth - 4) // 6
        ch = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]
        self.conv1 = nn.Conv2d(3, ch[0], 3, 1, 1, bias=False)
        self.group1 = self._make_group(ch[0], ch[1], n, stride=1, dropout=dropout)
        self.group2 = self._make_group(ch[1], ch[2], n, stride=2, dropout=dropout)
        self.group3 = self._make_group(ch[2], ch[3], n, stride=2, dropout=dropout)
        self.bn_final = nn.BatchNorm2d(ch[3])
        self.fc = nn.Linear(ch[3], num_classes)
        self._init_weights()

    def _make_group(self, in_ch, out_ch, n_blocks, stride, dropout):
        layers = [WideResBlock(in_ch, out_ch, stride, dropout)]
        for _ in range(1, n_blocks):
            layers.append(WideResBlock(out_ch, out_ch, 1, dropout))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.conv1(x)
        x = self.group1(x)
        x = self.group2(x)
        x = self.group3(x)
        x = F.relu(self.bn_final(x), inplace=True)
        x = F.adaptive_avg_pool2d(x, 1)
        x = torch.flatten(x, 1)
        return self.fc(x)


model = WideResNet(depth=DEPTH, widen_factor=WIDEN_FACTOR, dropout=DROPOUT, num_classes=NUM_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 2. Data

In [ ]:
class Cutout:
    def __init__(self, size):
        self.size = size
    def __call__(self, img):
        h, w = img.shape[1], img.shape[2]
        y, x = np.random.randint(h), np.random.randint(w)
        img[:, max(0,y-self.size//2):min(h,y+self.size//2), max(0,x-self.size//2):min(w,x+self.size//2)] = 0.0
        return img

MEAN, STD = (0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)

train_transform = T.Compose([
    T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
    T.AutoAugment(T.AutoAugmentPolicy.CIFAR10),
    T.ToTensor(), T.Normalize(MEAN, STD), Cutout(CUTOUT_SIZE),
])
test_transform = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

trainset = torchvision.datasets.CIFAR100(root='/content/data', train=True, download=True, transform=train_transform)
testset = torchvision.datasets.CIFAR100(root='/content/data', train=False, download=True, transform=test_transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(trainset)} | Test: {len(testset)}')

## 3. Training

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, nesterov=True)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

CKPT_PATH = os.path.join(DRIVE_DIR, 'last_checkpoint.pth')
LOG_PATH = os.path.join(DRIVE_DIR, 'training_log.json')

start_epoch = 0
best_acc = 0.0
history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    best_acc = ckpt['best_acc']
    history = ckpt['history']
    print(f'Resumed from epoch {start_epoch}, best_acc={best_acc:.2f}%')
else:
    print('Training from scratch.')


def train_one_epoch(loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * targets.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        total += targets.size(0)
    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        loss = criterion(outputs, targets)
        total_loss += loss.item() * targets.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        total += targets.size(0)
    return total_loss / total, 100.0 * correct / total


print(f'Starting training — epochs {start_epoch+1} to {EPOCHS}')
t0 = time.time()

for epoch in range(start_epoch, EPOCHS):
    lr_now = optimizer.param_groups[0]['lr']
    train_loss, train_acc = train_one_epoch(train_loader)
    val_loss, val_acc = evaluate(test_loader)
    scheduler.step()

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(round(train_loss, 5))
    history['train_acc'].append(round(train_acc, 2))
    history['val_loss'].append(round(val_loss, 5))
    history['val_acc'].append(round(val_acc, 2))
    history['lr'].append(round(lr_now, 6))

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(DRIVE_DIR, 'best_model.pth'))

    torch.save({
        'epoch': epoch, 'model': model.state_dict(),
        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
        'best_acc': best_acc, 'history': history,
    }, CKPT_PATH)

    elapsed = time.time() - t0
    eta = elapsed / (epoch - start_epoch + 1) * (EPOCHS - epoch - 1) if epoch > start_epoch else 0
    print(f'Epoch {epoch+1:>3d}/{EPOCHS} | lr={lr_now:.5f} | '
          f'train_loss={train_loss:.4f} acc={train_acc:.2f}% | '
          f'val_loss={val_loss:.4f} acc={val_acc:.2f}% | '
          f'best={best_acc:.2f}% | ETA={eta/60:.0f}min')

total_time = time.time() - t0
print(f'\nDone in {total_time/3600:.1f}h. Best val accuracy: {best_acc:.2f}%')
with open(LOG_PATH, 'w') as f:
    json.dump(history, f, indent=2)

## 4. Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['epoch'], history['train_loss'], label='Train')
ax1.plot(history['epoch'], history['val_loss'], label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(history['epoch'], history['train_acc'], label='Train')
ax2.plot(history['epoch'], history['val_acc'], label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Acc (%)'); ax2.set_title(f'Accuracy — Best: {best_acc:.2f}%'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 5. Summary

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
summary_text = f"""============================================
MODEL SUMMARY — WideResNet-{DEPTH}-{WIDEN_FACTOR} + SE Attention
============================================
Dataset:          CIFAR-100
Total params:     {total_params:,}
Epochs:           {EPOCHS}
Batch size:       {BATCH_SIZE}
Optimizer:        SGD (lr={LR}, momentum={MOMENTUM}, wd={WEIGHT_DECAY}, nesterov)
Scheduler:        CosineAnnealingLR
Augmentation:     RandomCrop + HFlip + AutoAugment + Cutout({CUTOUT_SIZE})
Label smoothing:  {LABEL_SMOOTHING}
Dropout:          {DROPOUT}
Best val acc:     {best_acc:.2f}%
============================================\n"""
print(summary_text)
with open(os.path.join(DRIVE_DIR, 'model_summary.txt'), 'w') as f:
    f.write(summary_text)
    f.write('\n--- Epoch log ---\n')
    for i in range(len(history['epoch'])):
        f.write(f"Epoch {history['epoch'][i]:>3d} | train_loss={history['train_loss'][i]:.5f} acc={history['train_acc'][i]:.2f}% | val_loss={history['val_loss'][i]:.5f} val_acc={history['val_acc'][i]:.2f}%\n")
print(f'All files saved to Google Drive: {DRIVE_DIR}')